In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

def load_data(item_type='total', cal_func='sum'):
    risk_scores = pd.read_csv(f'./Data/Fama_French/risk_scores_{item_type}_{cal_func}.csv') 
    daily_data = pd.read_csv('./Data/Fama_French/stock_daily_data.csv', parse_dates=['date'])
    factors = pd.read_csv('./Data/Fama_French/ff_factors.csv', parse_dates=['date'], index_col='date')
    return risk_scores, daily_data, factors

def run_portfolio_construction(risk_scores, daily_data, splita_num=5):
    rebalance_years = sorted(risk_scores['year'].unique())
    all_portfolio_returns = []
    for year in rebalance_years:
        formation_date = pd.to_datetime(f'{year+1}-06-30')
        holding_period_start = pd.to_datetime(f'{year+1}-07-01')
        holding_period_end = pd.to_datetime(f'{year+2}-06-30')
        current_scores = risk_scores[risk_scores['year'] == year].copy()
        current_scores.loc[:, 'portfolio'] = pd.qcut(current_scores['risk_score'], q=splita_num, precision=10, labels=False, duplicates='drop') + 1

        low_risk_portfolio_tickers = current_scores[current_scores['portfolio'] == 1]['ticker'].unique()
        high_risk_portfolio_tickers = current_scores[current_scores['portfolio'] == splita_num]['ticker'].unique()
        print(f"Year: {year}")
        print(f"Low Risk Portfolio Tickers: {low_risk_portfolio_tickers}")
        print(f"High Risk Portfolio Tickers: {high_risk_portfolio_tickers}")

        period_data = daily_data[
            (daily_data['date'] >= holding_period_start) & 
            (daily_data['date'] <= holding_period_end)
        ]

        data_with_portfolio = pd.merge(period_data, current_scores[['ticker', 'portfolio']], on='ticker', how='inner')
        daily_returns = data_with_portfolio.groupby(['date', 'portfolio'])['return'].mean().unstack()
        
        if 1 in daily_returns.columns and splita_num in daily_returns.columns:
            low_risk_returns = daily_returns[1]
            high_risk_retturns = daily_returns[splita_num]
            ls_returns =  low_risk_returns - high_risk_retturns
            period_results = pd.DataFrame({
                'Low Risk': low_risk_returns,
                f'High Risk': high_risk_retturns,
                'LS': ls_returns,
                'S&P 500 (Baseline)': period_data.groupby('date').agg(risk_score=('return', 'mean'))['risk_score'].values
            })
            # print(period_results)
            all_portfolio_returns.append(period_results)
    final_portfolios = pd.concat(all_portfolio_returns).dropna()
    return final_portfolios

def analyze_and_visualize(portfolios, save_path='figure_1_cumulative_returns.png'):
    trading_days = 252
    annual_return = portfolios.mean() * trading_days
    annual_volatility = portfolios.std() * np.sqrt(trading_days)
    sharpe_ratio = annual_return / annual_volatility # Simple Sharpe Ratio, assuming risk-free rate is 0
    
    # Calculate maximum drawdown
    cumulative_returns = (1 + portfolios).cumprod()
    peak = cumulative_returns.expanding(min_periods=1).max()
    drawdown = (cumulative_returns - peak) / peak
    max_drawdown = drawdown.min()
    performance_summary = pd.DataFrame({
        'Annualized Return': [f"{x:.2%}" for x in annual_return],
        'Annualized Volatility': [f"{x:.2%}" for x in annual_volatility],
        'Sharpe Ratio': [f"{x:.2f}" for x in sharpe_ratio],
        'Max Drawdown': [f"{x:.2%}" for x in max_drawdown]
    }, index=portfolios.columns)
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(12, 7))
    for col in cumulative_returns.columns:
        ax.plot(cumulative_returns.index, cumulative_returns[col], label=col)
    ax.set_title('Portfolio Cumulative Returns', fontsize=16)
    ax.set_ylabel('Cumulative Return', fontsize=12)
    ax.set_xlabel('Date', fontsize=12)
    ax.legend(title='Portfolio', fontsize=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(save_path, dpi=i)
    # plt.close()

def run_regression_analysis(ls_portfolio, factors):
    data_for_regression = pd.merge(ls_portfolio, factors, left_index=True, right_index=True, how='inner')
    Y = data_for_regression['LS']
    X = data_for_regression[['MKT-RF', 'SMB', 'HML']]
    X = sm.add_constant(X)
    model = sm.OLS(Y, X).fit()
    annualized_alpha = model.params['const'] * 252
    results_summary = pd.DataFrame({
        'Coefficient': model.params,
        't-statistic': model.tvalues
    })
    results_summary.loc['const', 'Coefficient'] = annualized_alpha
    results_summary = results_summary.rename(index={'const': 'Alpha (Annualized)'})
    alpha_t_stat = model.tvalues['const']
    return alpha_t_stat

# risk_scores, daily_data, factors = load_data('item_7', 'sum')
# portfolios = run_portfolio_construction(risk_scores, daily_data, splita_num=15)
# analyze_and_visualize(portfolios)
# analyze_and_visualize(portfolios)
# # run_regression_analysis(portfolios[['LS']], factors)

risk_scores, daily_data, factors = load_data('item_7', 'sum')
run_portfolio_construction(risk_scores, daily_data, splita_num=15)


Year: 2020
Low Risk Portfolio Tickers: ['ABBV' 'ACN' 'ADP' 'AKAM' 'AMD' 'BLDR' 'CAT' 'CCI' 'CME' 'CMG' 'CTVA'
 'FDS' 'GPC' 'KMI' 'LKQ' 'MCK' 'MCO' 'MKTX' 'MLM' 'MRNA' 'MU' 'NEE' 'O'
 'PARA' 'PKG' 'PLTR' 'PM' 'SPGI' 'SWKS' 'TROW' 'URI' 'VRTX' 'WST']
High Risk Portfolio Tickers: ['ANSS' 'APTV' 'BWA' 'CHTR' 'CL' 'CMCSA' 'CNP' 'EL' 'EMN' 'EXPD' 'GD'
 'GNRC' 'HII' 'HLT' 'HUBB' 'KMB' 'LII' 'LMT' 'LNT' 'MA' 'PRU' 'RL' 'RTX'
 'SLB']
Year: 2021
Low Risk Portfolio Tickers: ['AMD' 'ATO' 'BFB' 'CCI' 'CPB' 'CSGP' 'DHI' 'DUK' 'FDS' 'GLW' 'MO' 'MPWR'
 'NOW' 'NUE' 'PODD' 'SBAC' 'SNA' 'SRE' 'STE' 'TDY' 'TROW' 'UBER' 'WST'
 'XEL']
High Risk Portfolio Tickers: ['ALL' 'AOS' 'APTV' 'CHRW' 'CINF' 'DG' 'ED' 'EL' 'EMN' 'EW' 'EXPE' 'GD'
 'GL' 'KMB' 'LLY' 'NKE' 'PEP' 'PKG' 'RSG' 'RTX' 'TER' 'UNH']
Year: 2022
Low Risk Portfolio Tickers: ['ADP' 'AMT' 'CHRW' 'CHTR' 'CPRT' 'EFX' 'EPAM' 'EXC' 'FI' 'LULU' 'LYB'
 'MKTX' 'NDSN' 'PODD' 'SBUX' 'SLB' 'SNA' 'TDG' 'UDR' 'VZ' 'WST']
High Risk Portfolio Tickers: []
Year: 2023

,Low Risk,High Risk,LS,S&P 500 (Baseline)
date,,,,
2021-07-01,0.001561,0.006500,-0.004939,0.006265
2021-07-02,0.003400,0.004499,-0.001099,0.003286
2021-07-06,-0.004041,-0.009372,0.005331,-0.006233
2021-07-07,-0.002912,0.003255,-0.006168,0.003368
2021-07-08,-0.005777,-0.009827,0.004050,-0.009852
...,...,...,...,...
2023-06-26,0.008423,0.005522,0.002900,0.005009
2023-06-27,0.011270,0.013477,-0.002207,0.012418
2023-06-28,-0.001930,-0.008009,0.006079,-0.001587


In [4]:
item_types = ['item_7']
cal_funcs = ['sum']


df_result = pd.DataFrame(columns=['alpha_t', 'item', 'cal_func', 'split_num'])

for i_t in (item_types):
    for c_f in (cal_funcs):
        for i in (range(2, 150)):
            try:
                risk_scores, daily_data, factors = load_data(i_t, c_f)
                portfolios = run_portfolio_construction(risk_scores, daily_data, splita_num=i)
                # analyze_and_visualize(portfolios)
                alpha_t = run_regression_analysis(portfolios[['LS']], factors)
                print(i,'\t',alpha_t)
                df_result.loc[len(df_result)] = [alpha_t, i_t, c_f, i]
            except Exception as e:
                continue

2 	 0.3262449674678392
3 	 0.49361082524202987
4 	 0.947552357065496
5 	 1.7766833637130657
6 	 1.4191528795135757
7 	 1.1827332963530266
8 	 1.3079323238246556
9 	 1.2898877048891353
10 	 1.2887908380017532
11 	 0.770304672075353
12 	 0.7347212033571544
13 	 0.62884209897271
14 	 0.6017513255900901
15 	 1.7895666203090956
16 	 1.1429315265200366
17 	 2.5641450146612987
18 	 2.416939572730954
19 	 2.2828517164228486
20 	 2.2828517164228486
21 	 2.1363156373743184
22 	 1.9170560287680614
23 	 1.3249512813208149
24 	 1.3249512813208149
25 	 1.9368506625082853
26 	 1.9368506625082853
27 	 1.9368506625082853
29 	 1.9368506625082853
